# batchnorm-affine-params — faded example 3: Fold BatchNorm affine scale into a per-channel factor

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `batchnorm-affine-params`. Running the beacon reports progress on the `CNN: BatchNorm affine params` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: BatchNorm affine params` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`batchnorm-affine-params`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "batchnorm-affine-params"
DD_SUBTOPIC = "CNN: BatchNorm affine params"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Because BatchNorm's affine `y = gamma * x_hat + beta` is per-channel and linear, a constant per-channel pre-scale `s` can be folded directly into the effective scale: applying scale `s[c]` then `gamma[c]` is the same as applying `gamma[c] * s[c]`. This is the algebra behind BN-into-conv fusion.

## Faded exercise 3

### Faded — fold a pre-scale into BatchNorm's gamma

Implement `bn_affine_with_prescale(x_hat, gamma, beta, s)`. The input is first scaled per-channel by `s: (C,)`, then the BatchNorm affine `y = gamma * (s * x_hat) + beta` is applied. Rather than two passes, fold `s` into the effective scale and apply it once.

The broadcasting reshape and final formula are given. You must complete the **effective per-channel scale**.

**Fill in:** Compute the fused effective scale eff = gamma * s (elementwise over channels) before reshaping for broadcast.

In [ ]:
def bn_affine_with_prescale(x_hat: Tensor, gamma: Tensor, beta: Tensor, s: Tensor) -> Tensor:
    eff = None  # TODO: Compute the fused effective scale eff = gamma * s elementwise over channels.
    g = eff.view(1, -1, 1, 1)
    b = beta.view(1, -1, 1, 1)
    return g * x_hat + b


def _test():
    t.manual_seed(0)
    B, C, H, W = 2, 3, 3, 3
    x_hat = t.randn(B, C, H, W)
    gamma = t.tensor([2.0, 0.5, 1.0])
    beta = t.tensor([1.0, -1.0, 0.0])
    s = t.tensor([0.5, 4.0, 2.0])
    out = bn_affine_with_prescale(x_hat, gamma, beta, s)
    assert out.shape == (B, C, H, W), out.shape
    # Ground truth: two explicit passes
    pre = s.view(1, -1, 1, 1) * x_hat
    ref = gamma.view(1, -1, 1, 1) * pre + beta.view(1, -1, 1, 1)
    assert t.allclose(out, ref, atol=1e-6), 'fused scale mismatch'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def bn_affine_with_prescale(x_hat: Tensor, gamma: Tensor, beta: Tensor, s: Tensor) -> Tensor:
    eff = gamma * s
    g = eff.view(1, -1, 1, 1)
    b = beta.view(1, -1, 1, 1)
    return g * x_hat + b
```
</details>